<a href="https://colab.research.google.com/github/bh1rg1v/colab/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [ ]:
!apt install python3-libtorrent

import libtorrent as lt

ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []

### Mount Google Drive
To stream files we need to mount Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### Add From Torrent File
You can run this cell to add more files as many times as you want

In [ ]:
from google.colab import files

source = files.upload()
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)


In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor

# @title Fast Folder Deletion Settings
folder_to_clean_fast = "/content/drive/MyDrive/Torrent"
max_workers = 20

def delete_item(file_path):
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
        return f"Deleted: {os.path.basename(file_path)}"
    except Exception as e:
        return f"Failed to delete {os.path.basename(file_path)}. Reason: {e}"

def fast_delete_folder_contents(folder_path, num_workers):
    if not os.path.exists(folder_path):
        print(f"The directory '{folder_path}' does not exist.")
        return

    # confirm = input(f"Are you sure you want to delete ALL files and folders inside '{folder_path}' in parallel? (yes/no): ")
    # if confirm.lower() != 'yes':
    #     print("Deletion cancelled.")
    #     return

    print("Gathering file list...")
    items = [os.path.join(folder_path, name) for name in os.listdir(folder_path)]

    if not items:
        print("No items to delete.")
        return

    print(f"Starting concurrent deletion of {len(items)} items using {num_workers} workers...")
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = executor.map(delete_item, items)
        for r in results:
            if r:
                print(r)

    print("Cleanup completed!")

fast_delete_folder_contents(folder_to_clean_fast, max_workers)

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

# New Code Starts Here

In [ ]:
!python -m pip install --no-cache-dir libtorrent==2.0.15

import libtorrent as lt

print("Version:", lt.version)

ses = lt.session()
ses.listen_on(6881, 6891)

downloads = []

print("SUCCESS")

In [7]:
from google.colab import files
import os

print("Upload your torrent file below:")
uploaded = files.upload()
torrent_path = list(uploaded.keys())[0]
print(f"Uploaded torrent: {torrent_path}")

Upload your torrent file below:


Saving reddit-ba051999301b109eab37d16f027b3f49ade2de13.torrent to reddit-ba051999301b109eab37d16f027b3f49ade2de13.torrent
Uploaded torrent: reddit-ba051999301b109eab37d16f027b3f49ade2de13.torrent


In [8]:
import os
import libtorrent as lt

# Load torrent info
t_info = lt.torrent_info(torrent_path)

# Gather file info and sort by file name
files_list = []
for i in range(t_info.num_files()):
    f = t_info.file_at(i)
    files_list.append({
        'index': i,
        'path': f.path,
        'size': f.size
    })

# Sort files alphabetically based on path/name
files_sorted = sorted(files_list, key=lambda x: x['path'])

# Calculate batches (200GB = 200 * 1024 * 1024 * 1024 bytes)
BATCH_LIMIT = 200 * 1024 * 1024 * 1024
current_batch_size = 0
batches = []
current_batch = []

for file in files_sorted:
    if current_batch_size + file['size'] > BATCH_LIMIT and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_batch_size = 0
    current_batch.append(file)
    current_batch_size += file['size']
if current_batch:
    batches.append(current_batch)

# Write details to /content/torrent_files_list.txt
output_path = "/content/torrent_files_list.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("=== ALL FILES (SORTED BY NAME) ===\n")
    for idx, item in enumerate(files_sorted):
        f.write(f"No: {idx} | Original Index: {item['index']} | Size: {item['size'] / (1024**2):.2f} MB | Name: {item['path']}\n")

    f.write("\n=== BATCHES (200GB LIMIT) ===\n")
    print("\n--- Torrent Batches (Sorted Alphabetically) ---")
    for b_idx, batch in enumerate(batches):
        start_file_num = files_sorted.index(batch[0])
        end_file_num = files_sorted.index(batch[-1])
        batch_size_gb = sum(item['size'] for item in batch) / (1024**3)
        batch_info = f"Batch {b_idx + 1}: Start Sorted No = {start_file_num}, End Sorted No = {end_file_num} (Total Size: {batch_size_gb:.2f} GB)"
        print(batch_info)
        f.write(batch_info + "\n")

print(f"\nSaved full list and batch layout to {output_path}")


--- Torrent Batches (Sorted Alphabetically) ---
Batch 1: Start Sorted No = 0, End Sorted No = 123 (Total Size: 199.72 GB)
Batch 2: Start Sorted No = 124, End Sorted No = 151 (Total Size: 193.12 GB)
Batch 3: Start Sorted No = 152, End Sorted No = 165 (Total Size: 185.12 GB)
Batch 4: Start Sorted No = 166, End Sorted No = 177 (Total Size: 196.10 GB)
Batch 5: Start Sorted No = 178, End Sorted No = 187 (Total Size: 194.98 GB)
Batch 6: Start Sorted No = 188, End Sorted No = 195 (Total Size: 180.21 GB)
Batch 7: Start Sorted No = 196, End Sorted No = 203 (Total Size: 195.87 GB)
Batch 8: Start Sorted No = 204, End Sorted No = 210 (Total Size: 187.76 GB)
Batch 9: Start Sorted No = 211, End Sorted No = 217 (Total Size: 198.26 GB)
Batch 10: Start Sorted No = 218, End Sorted No = 223 (Total Size: 181.40 GB)
Batch 11: Start Sorted No = 224, End Sorted No = 350 (Total Size: 198.45 GB)
Batch 12: Start Sorted No = 351, End Sorted No = 409 (Total Size: 198.50 GB)
Batch 13: Start Sorted No = 410, End S

/tmp/ipykernel_840/3228585513.py:10: DeprecationWarning: file_at() is deprecated
  f = t_info.file_at(i)
/tmp/ipykernel_840/3228585513.py:14: DeprecationWarning: file_entry is deprecated
  'size': f.size


In [9]:
from google.colab import drive
import os

drive.mount("/content/drive")
download_path = "/content/drive/My Drive/Torrent"
os.makedirs(download_path, exist_ok=True)

Mounted at /content/drive


In [13]:
# Change these numbers to choose which sorted files to download
# Defaults to the first batch
start = 0
end = 123

# try:
#     # Automatically resolve default ending for the first batch if calculated
#     end = files_sorted.index(batches[0][-1])
# except:
#     pass

print(f"Configured sorted file range: Start = {start}, End = {end}")

Configured sorted file range: Start = 0, End = 123


In [ ]:
import sys
import time
import os

# Set download priorities: 1 for selected range, 0 for others
priorities = [0] * t_info.num_files()

selected_files = files_sorted[start:end+1]
for item in selected_files:
    priorities[item['index']] = 1

# Track the local destination of files being downloaded in this cell
downloaded_file_paths = []
for item in selected_files:
    full_dest_path = os.path.join(download_path, item['path'])
    downloaded_file_paths.append(full_dest_path)

# Write the paths of the downloaded files to a text file for later retrieval
recent_downloads_log = "/content/recently_downloaded.txt"
with open(recent_downloads_log, "w", encoding="utf-8") as f:
    for path in downloaded_file_paths:
        f.write(path + "\n")

# Add Torrent with file priorities
params = {
    "save_path": download_path,
    "ti": t_info,
    "file_priorities": priorities
}
handle = ses.add_torrent(params)
downloads.append(handle)

print("Downloading selected batch files...")
while not handle.is_seed():
    s = handle.status()
    print(f"\rProgress: {s.progress * 100:.2f}% | Down: {s.download_rate / 1000:.2f} kB/s | State: {s.state}", end="")
    sys.stdout.flush()
    time.sleep(2)
print(f"\nFinished downloading target range [{start} to {end}]. File list saved to {recent_downloads_log}")

Progress: 0.00% | Down: 0.00 kB/s | State: checking_resume_data

/tmp/ipykernel_840/3949642767.py:34: DeprecationWarning: is_seed() is deprecated
  while not handle.is_seed():


Progress: 9.55% | Down: 4739.64 kB/s | State: downloading

In [ ]:
import os
import shutil

# Load recently downloaded paths from the log file and delete them
recent_downloads_log = "/content/recently_downloaded.txt"

if os.path.exists(recent_downloads_log):
    with open(recent_downloads_log, "r", encoding="utf-8") as f:
        paths_to_delete = [line.strip() for line in f if line.strip()]

    print(f"Loaded {len(paths_to_delete)} paths to clean up from {recent_downloads_log}...")
    for path in paths_to_delete:
        if os.path.exists(path):
            try:
                if os.path.isdir(path):
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                print(f"Successfully removed: {path}")
            except Exception as e:
                print(f"Failed to remove {path}: {e}")
        else:
            print(f"File or directory already absent: {path}")

    # Clear the log file content after deletion
    open(recent_downloads_log, 'w').close()
    print("Cleanup completed. Log file cleared.")
else:
    print(f"No log file found at {recent_downloads_log}. Nothing to delete.")